# ⚡ Interactive Tutorial: MCP Circuit Sandbox

Welcome to the interactive Jupyter Notebook tutorial for the **MCP Circuit Sandbox**! Because this is a notebook, you can run the code cells directly in your editor to see how it works.

## Step 1: The Raw Electrical Engineering Logic

Before we even talk about MCP, let's look at the underlying Python logic. We have pure Python functions that calculate electrical properties. Run the cell below to test Ohm's Law!

In [ ]:
from ee_mcp_demo.circuits import calculate_ohms_law, calculate_electrical_power

# Calculate Current (I) given Voltage (V) and Resistance (R)
ohms_result = calculate_ohms_law(voltage=12, resistance=6)
print(f"Ohm's Law Result: {ohms_result}")

# Calculate Power (P) given Voltage (V) and Current (I)
power_result = calculate_electrical_power(voltage=12, current=2)
print(f"Power Result: {power_result}")

## Step 2: The MCP Server

In `src/ee_mcp_demo/server.py`, we wrapped those functions using `@mcp.tool()`. This turns them into tools that an AI agent (like me!) can call.

To see this visually, you can use the **MCP Inspector**. Since the Inspector runs a web server, you should run this command in a **separate terminal window** so it doesn't freeze this notebook:

```bash
uv run mcp dev src/ee_mcp_demo/server.py
```

Once running, you can open the browser URL it gives you to test the tools manually!

## Step 3: Connecting to the Server via Python (Client)

What if we want to write a Python script that connects to our MCP server and asks it for its tools? The code below uses the official MCP Python SDK to spin up our server in the background and communicate with it over `stdio`.

*(Note: When running subprocesses from Jupyter, we need to pass `errlog=sys.__stderr__` to bypass a limitation with Jupyter's output streams).* 

In [ ]:
import asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_mcp_client():
    # 1. Define how to start our server
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["-m", "ee_mcp_demo.server"],
    )

    # 2. Connect to the server
    print("Connecting to MCP Server...")
    async with stdio_client(server_params, errlog=sys.__stderr__) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            print("Connected!\n")
            
            # 3. Ask the server what tools it has
            tools = await session.list_tools()
            print("🛠️ Available Tools:")
            for tool in tools.tools:
                print(f" - {tool.name}: {tool.description}")
                
            # 4. Call a tool through the protocol!
            print("\n⚡ Calling 'calculate_parallel_resistance'...")
            result = await session.call_tool(
                "calculate_parallel_resistance",
                arguments={"resistances": [100, 100]}
            )
            print(f"Result: {result.content}")

# Run the async client
await run_mcp_client()

## Step 4: Interactive UI & New Laws!

We've added **Coulomb's Law** and **Capacitance** tools to the MCP server. Below is an interactive UI using `ipywidgets` that lets you play with these formulas without writing any code!

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ee_mcp_demo.circuits import calculate_coulombs_law, calculate_charge

out = widgets.Output()

# 1. Coulomb's Law UI
q1_slider = widgets.FloatLogSlider(value=1e-6, base=10, min=-9, max=-3, description='Q1 (C):')
q2_slider = widgets.FloatLogSlider(value=1e-6, base=10, min=-9, max=-3, description='Q2 (C):')
r_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Dist (m):')
calc_coulomb_btn = widgets.Button(description="Calculate Force", button_style='info')

def on_coulomb_click(b):
    with out:
        out.clear_output()
        result = calculate_coulombs_law(q1_slider.value, q2_slider.value, r_slider.value)
        print(f"⚡ Electrostatic Force: {result['force']:.4e} {result['unit']}")

calc_coulomb_btn.on_click(on_coulomb_click)
coulomb_ui = widgets.VBox([widgets.HTML("<b>Coulomb's Law Calculator</b>"), q1_slider, q2_slider, r_slider, calc_coulomb_btn])

# 2. Capacitance UI
cap_slider = widgets.FloatLogSlider(value=1e-3, base=10, min=-6, max=0, description='Cap (F):')
volt_slider = widgets.FloatSlider(value=12.0, min=1.0, max=100.0, description='Volt (V):')
calc_cap_btn = widgets.Button(description="Calculate Charge", button_style='success')

def on_cap_click(b):
    with out:
        out.clear_output()
        result = calculate_charge(cap_slider.value, volt_slider.value)
        print(f"🔋 Stored Charge: {result['charge']:.4e} {result['unit']}")

calc_cap_btn.on_click(on_cap_click)
cap_ui = widgets.VBox([widgets.HTML("<br><b>Capacitor Charge Calculator</b>"), cap_slider, volt_slider, calc_cap_btn])

display(widgets.HBox([coulomb_ui, cap_ui]))
display(out)